Import Libraries

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
# from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from dotenv import load_dotenv

load_dotenv()
print("Imports all Successfull!!!")

Define AgentState

In [ ]:
class AgentState(TypedDict):
    infrastructure: str
    policy: str
    critique: str
    rounds: int

print("State Defined!!!")

Initialize Claude

In [ ]:
llm = ChatAnthropic(
    model = "claude-sonnet-4-6",
    temperature = 0
)

print("Claude is ready!!!")

Policy Generator

In [ ]:
def generate_policy(state: AgentState) -> dict:
    rounds = state.get("rounds", 0) + 1
    print(f"\n🔄 Generation round {rounds}...")
    
    prompt = state["infrastructure"]
    critique = state.get("critique", "")
    
    # First round - generate from scratch
    # Subsequent rounds - improve based on critique
    if critique:
        user_content = f"""
Infrastructure Description:
{prompt}

Previous Security Review Critique:
{critique}

Based on the critique above, generate an improved and comprehensive 
security policy that addresses all identified gaps and weaknesses.
"""
    else:
        user_content = f"""
Infrastructure Description:
{prompt}

Generate a comprehensive security policy for this infrastructure.
"""
    
    response = llm.invoke([
        SystemMessage(content="""
You are a senior cloud security architect with 20+ years of experience 
across Azure, AWS and Google Cloud Platform.

Generate a structured security policy covering:
1. Identity & Access Management (IAM)
2. Network Security & Segmentation  
3. Data Protection & Classification
4. Secret & Credential Management
5. Logging, Monitoring & Alerting
6. Patch & Vulnerability Management
7. Incident Response
8. Compliance Considerations

Be specific, actionable and concise. Maximum 400 words.
        """),
        HumanMessage(content=user_content)
    ])
    
    print(f"✅ Generation {rounds} complete")
    return {
        "policy": response.content,
        "rounds": rounds
    }

print("Policy Generator node defined!")

Policy Reflector

In [ ]:
def reflect_on_policy(state: AgentState) -> dict:
    print(f"\n🔍 Reflecting on policy...")
    
    policy = state["policy"]
    infrastructure = state["infrastructure"]
    
    response = llm.invoke([
        SystemMessage(content="""
You are an experienced multi-cloud security policy reviewer and auditor
with deep expertise in enterprise security frameworks including:
- OWASP Top 10
- CIS Benchmarks
- NIST Cybersecurity Framework
- ISO 27001
- PCI-DSS and SOC2 compliance

Your job is to critically review security policies and identify:
1. Missing security controls
2. Weak or vague recommendations  
3. Compliance gaps
4. Implementation risks
5. Specific improvements needed

If the policy is comprehensive and production-ready, 
start your response with exactly 'SATISFACTORY' followed by your review.
If gaps exist, list them clearly with specific remediation steps.
Maximum 300 words.
        """),
        HumanMessage(content=f"""
Please review this security policy for the following infrastructure:

Infrastructure: {infrastructure}

Security Policy to Review:
{policy}

Provide your critique and recommendations.
        """)
    ])
    
    print(f"✅ Reflection complete")
    print(f"   SATISFACTORY detected: {'SATISFACTORY' in response.content}")
    return {
        "critique": response.content
    }

print("Policy Reflector node defined!")

Conditional Edge

In [ ]:
def should_continue(state: AgentState) -> str:
    print(f"\n⚡ Checking condition - rounds: {state.get('rounds',0)}")
    if "SATISFACTORY" in state["critique"] or state.get("rounds", 0) >= 2:
        print("→ Ending")
        return "end"
    print("→ Reflecting")
    return "reflect"

Build the Graph

In [ ]:
workflow = StateGraph(AgentState)
workflow.add_node("generate_policy", generate_policy)
workflow.add_node("reflect_on_policy", reflect_on_policy)
workflow.add_edge("reflect_on_policy", "generate_policy")
workflow.add_conditional_edges("generate_policy", should_continue, {
    "reflect" : "reflect_on_policy",
    # print("Current n:" state["messages"])
    "end": END,
})
workflow.set_entry_point("generate_policy")
print("Graph is built!!!")

Compile and Invoke

In [ ]:
app = workflow.compile()
result = app.invoke({
    # "infrastructure": "Create EC2 instances in DEV environment and restrict access to the development team only",
    # "infrastructure": "Create Azure Data Factory Instance in DEV, QA, PROD:,
    "infrastructure": "Kubernetes cluster implementation for DEV & QA environments for Azure, AWS & GCP",
    "policy": "",
    "critique": "",
    "rounds": 0
})

print(f"\n✅ Completed in {result['rounds']} rounds")
print("\n📋 Final Security Policy:")
print("=" * 60)
print(result["policy"])
print("\n🔍 Final Critique:")
print("=" * 60)
print(result["critique"])

Print Check

In [ ]:
print(f"Rounds: {result['rounds']}")
print(f"\nPolicy empty: {result['policy'] == ''}")
print(f"\nCritique empty: {result['critique'] == ''}")
print(f"\nPolicy preview: {result['policy'][:200]}")
print(f"\nCritique preview: {result['critique'][:200]}")